# 12 Quantum Portfolio Risk Engine

Price multiple options, aggregate quantum portfolio value and Greeks, and compare with classical portfolio outputs.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
positions = load_portfolio_config()
classical_df, classical_totals = value_portfolio(positions)
quantum_df, quantum_totals = value_portfolio_quantum(positions, n_qubits=int(config["quantum"]["portfolio_grid_qubits"]), x_width=float(config["quantum"]["x_width"]))
comparison = classical_df[["label", "theoretical_price", "position_theoretical_value"]].merge(
    quantum_df[["label", "quantum_price", "position_quantum_value", "post_selection_probability"]], on="label")
comparison["price_error"] = comparison["quantum_price"] - comparison["theoretical_price"]
comparison["value_error"] = comparison["position_quantum_value"] - comparison["position_theoretical_value"]
portfolio_value_error = quantum_totals["total_quantum_value"] - classical_totals["total_theoretical_value"]
assert abs(classical_df["position_theoretical_value"].sum() - classical_totals["total_theoretical_value"]) < 1e-8
assert abs(quantum_df["position_quantum_value"].sum() - quantum_totals["total_quantum_value"]) < 1e-8
print("VALIDATION PASSED: classical and quantum portfolio values equal sums of position values")
save_table(comparison, "12_quantum_portfolio_value_comparison.csv")
greek_errors = {}
for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
    greek_errors[greek] = quantum_totals[f"portfolio_quantum_{greek}"] - classical_totals[f"portfolio_{greek}"]
summary = {"classical_totals": classical_totals, "quantum_totals": quantum_totals, "portfolio_value_error": portfolio_value_error, "portfolio_greeks_error": greek_errors}
save_output(summary, "12_quantum_portfolio_summary.json")
plt.figure()
plt.bar(comparison["label"], comparison["value_error"])
plt.xticks(rotation=45, ha="right")
plt.title("Position-level quantum value error")
plt.ylabel("Quantum - classical value")
save_current_figure("12_quantum_portfolio_value_errors.png")
comparison
